In [1]:
from conflict_monitoring_ntl.case_studies import get_county_ids, get_date
from conflict_monitoring_ntl.satellites import (
    BlackMarblePy, 
    GHSLSurface, 
    BlackMarbleEE,
    GHSLPopulation,
)
from conflict_monitoring_ntl.utils import get_gdf_for_admin, binarize_xarray
from conflict_monitoring_ntl.transform import RasterPipeline

from rasterio.enums import Resampling

import xarray as xr
import holoviews as hv
import hvplot.xarray  # noqa

In [2]:
country = "Sudan"
date, county_id = get_date(country), get_county_ids(country)[0]
gdf = get_gdf_for_admin(county_id)

In [ ]:
rasters = [
    GHSLPopulation(),
    BlackMarblePy(frequency="daily"), 
    BlackMarblePy(frequency="monthly"), 
    BlackMarblePy(frequency="annual"), 
]

transformations = [
    {"reproject_match": {"resampling": Resampling.sum}}, {}, {}, {}, 
]
pipeline = RasterPipeline(gdf, date, rasters, transformations)
ds = pipeline.run()

In [ ]:
opts = {
    "labelled": [],
    "xticks": [],
    "yticks": []
}

daily_ds = ds.black_marble_radiance_daily
daily = ds.black_marble_radiance_daily.hvplot.image(
    x="lon",
    y="lat",
    crs=daily_ds.rio.crs,
    clim=(0, 2),
    cmap="inferno",
    title="Black Marble - daily",
    geo=True,
).opts(labelled=[])

monthly = ds.black_marble_radiance_monthly.hvplot.image(
    x="lon",
    y="lat",
    crs=daily_ds.rio.crs,
    clim=(0, 2),
    cmap="inferno",
    title="Black Marble - monthly",
    geo=True,
).opts(labelled=[])

annual = ds.black_marble_radiance_annual.hvplot.image(
    x="lon",
    y="lat",
    crs=daily_ds.rio.crs,
    clim=(0, 2),
    cmap="inferno",
    title="Black Marble - annual",
    geo=True,
).opts(labelled=[])

ghsl = ds.ghsl_population.hvplot.image(
    x="lon",
    y="lat",
    crs=daily_ds.rio.crs,
    clim=(0, 500),
    cmap="viridis",
    title="GHSL Population",
    geo=True,
).opts(labelled=[])

plot = (daily + monthly + annual + ghsl).cols(2)
plot

## Winterthur

In [ ]:
import datetime
import geopandas as gpd
import pygadm


date = datetime.date(2022, 3, 22)
gdf = pygadm.Items(name="Winterthur", content_level=2)
gdf = gpd.GeoDataFrame(geometry=gdf.geometry)
gdf = gdf.set_crs("EPSG:4326")

In [ ]:
frequencies = ["daily", "monthly", "annual"]
rasters = [BlackMarblePy(frequency=f).raster(gdf, date) for f in frequencies]

ds = xr.merge(rasters).rio.write_crs("EPSG:4326")

In [ ]:
ds

In [ ]:
daily_ds = ds.black_marble_radiance_daily
daily = ds.black_marble_radiance_daily.hvplot.image(
    x="lon",
    y="lat",
    crs=daily_ds.rio.crs,
    clim=(0, 2),
    cmap="inferno",
    title="Daily",
    geo=True,
    colorbar=False,
)

monthly = ds.black_marble_radiance_monthly.hvplot.image(
    x="lon",
    y="lat",
    crs=daily_ds.rio.crs,
    clim=(0, 2),
    cmap="inferno",
    title="monthly",
    geo=True,
    colorbar=False
)

annual = ds.black_marble_radiance_annual.hvplot.image(
    x="lon",
    y="lat",
    crs=daily_ds.rio.crs,
    clim=(0, 2),
    cmap="inferno",
    title="annual",
    geo=True,
    colorbar=False
)

plot = daily + monthly + annual
plot

In [ ]:
hv.save(plot, 'output.html', backend='bokeh')

In [20]:
import pandas as pd

date_range = pd.date_range(start="2023-12-02", end="2023-12-04")
ds = BlackMarblePy("daily", drop_values_by_quality_flag=[2, 255]).raster(gdf, date_range)

OBTAINING MANIFEST...:   0%|          | 0/1 [00:00<?, ?it/s]

2025-11-05 10:32:06,928 - httpx - INFO - HTTP Request: GET https://ladsweb.modaps.eosdis.nasa.gov/api/v1/files?product=VNP46A2&collection=5200&dateRanges=2023-12-02..2023-12-04&areaOfInterest=x24.96y11.45%2Cx26.09y12.97 "HTTP/1.1 200 OK"


QUEUEING TASKS | Downloading (110.2 MB)...:   0%|          | 0/3 [00:00<?, ?file/s]

PROCESSING TASKS | Downloading (110.2 MB)...:   0%|          | 0/3 [00:00<?, ?file/s]

COLLECTING RESULTS | Downloading (110.2 MB)...:   0%|          | 0/3 [00:00<?, ?file/s]

COLLATING TILES | Processing...:   0%|          | 0/3 [00:00<?, ?date/s]

In [51]:
default_kwargs = {
    "x": "lon",
    "y": "lat",
    "crs": ds.rio.crs,
    "clim": (0, 2),
    "cmap": "inferno",
    "geo": True,
    "colorbar": False,
    "frame_width": 250,
    "aspect": 1,
}

In [54]:
first = ds.isel(time=-1).black_marble_radiance_daily.hvplot.image(
    title="Black Marble - 2023-12-04 \n*no fill",
    **default_kwargs
)

second_ds = ds.isel(time=slice(1, 3)).ffill(dim="time")
second = second_ds.isel(time=-1).black_marble_radiance_daily.hvplot.image(
    title="Black Marble - 2023-12-04 \n*filled with T-1",
    **default_kwargs
)

third_ds = ds.ffill(dim="time")
third = third_ds.isel(time=-1).black_marble_radiance_daily.hvplot.image(
    title="Black Marble - 2023-12-04 \n*filled with T-1 + T-2",
    **default_kwargs
)

first + second + third

:Layout
   .Image.I   :Image   [lon,lat]   (black_marble_radiance_daily)
   .Image.II  :Image   [lon,lat]   (black_marble_radiance_daily)
   .Image.III :Image   [lon,lat]   (black_marble_radiance_daily)